# Interactive Protein Intervention Notebook

This notebook provides an interactive interface for:
1. Selecting protein sequences and concepts
2. Performing concept-based interventions
3. Computing edit distances and biological metrics
4. Visualizing intervention effects

## Setup

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import Levenshtein
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio import pairwise2
from Bio.pairwise2 import format_alignment

# Lobster imports
from lobster.model import LobsterCBMPMLM

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Load Concept Bottleneck Model

Load the model for concept-based interventions:

In [ ]:
# Device selection
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Model checkpoint
model_checkpoint = widgets.Text(
    value='asalam91/cb_lobster_24M',
    description='Checkpoint:',
    layout=widgets.Layout(width='50%')
)

load_button = widgets.Button(
    description='Load Model',
    button_style='primary',
    icon='download'
)

model_status = widgets.Output()

model = None

def load_model_callback(b):
    global model
    with model_status:
        clear_output()
        print(f"Loading model: {model_checkpoint.value}...")
        try:
            model = LobsterCBMPMLM(model_checkpoint.value).to(device)
            model.eval()
            print("✓ Model loaded successfully!")
            
            total_params = sum(p.numel() for p in model.parameters())
            print(f"Total parameters: {total_params:,}")
            print(f"Available concepts: {len(model.list_supported_concept())}")
        except Exception as e:
            print(f"❌ Error loading model: {e}")

load_button.on_click(load_model_callback)

display(model_checkpoint)
display(load_button)
display(model_status)

## 2. Select Protein Sequence

Choose the protein sequence to modify:

In [ ]:
# Sequence source selection
seq_source = widgets.RadioButtons(
    options=['Direct Input', 'Example Sequences', 'FASTA File'],
    value='Example Sequences',
    description='Source:'
)

# Example sequences
example_sequences = {
    'G-protein alpha subunit': 'MGAGASAEEKHSRELEKKLKEDAEKDARTVKLLLLGAGESGKSTIVKQMKIIHQDGYSLEECLEFIAIIYGNTLQSILAIVRAMTTLNIQYGDSARQDDARKLMHMADTIEEGTMPKEMSDIIQRLWKDSGIQACFERASEYQLNDSAGYYLSDLERLVTPGYVPTEQDVLRSRVKTTGIIETQFSFKDLNFRMFDVGGQRSERKKWIHCFEGVTCIIFIAALSAYDMVLVEDDEVNRMHESLHLFNSICNHRYFATTSIVLFLNKKDVFFEKIKKAHLSICFPDYDGPNTYEDAGNYIKVQFLELNMRRDVKEIYSHMTCATDTQNVKFVFDAVTDIIIKENLKDCGLF',
    'Small peptide': 'MKFLKFSLLTAVLLSVVFAFSSCGDDDDTISSSTTGPPSPDLSRIVGGWECELGDNMECFTFKYGGCMGIGNRNNNFKTEECL',
    'Antibody VH': 'QVQLVQSGAEVKKPGASVKVSCKASGYTFTDYYMHWVRQAPGQGLEWMGWINPNSGGTNYAQKFQGRVTMTRDTSISTAYMELSRLRSDDTAVYYCAR',
    'Hydrophobic protein': 'MLLLLLLLLLLVVVVVVVVVIIIIIIIIIIFFFFFFF',
    'Charged protein': 'MKKKKKKKKKKKRRRRRRRRRREEEEEEEEEEDDDDDDDDDD'
}

example_dropdown = widgets.Dropdown(
    options=list(example_sequences.keys()),
    description='Example:'
)

# Direct input
direct_input = widgets.Textarea(
    value='',
    description='Sequence:',
    layout=widgets.Layout(width='80%', height='100px')
)

# FASTA file
fasta_path = widgets.Text(
    value='test_data/query.fasta',
    description='FASTA Path:'
)

load_seq_button = widgets.Button(
    description='Load Sequence',
    button_style='info',
    icon='check'
)

seq_status = widgets.Output()

test_protein = None
seq_id = None

def load_sequence(b):
    global test_protein, seq_id
    with seq_status:
        clear_output()
        
        if seq_source.value == 'Example Sequences':
            test_protein = example_sequences[example_dropdown.value]
            seq_id = example_dropdown.value
            print(f"✓ Loaded: {seq_id}")
        
        elif seq_source.value == 'Direct Input':
            test_protein = direct_input.value.strip().replace(' ', '').replace('\n', '')
            seq_id = 'user_sequence'
            print(f"✓ Loaded user sequence")
        
        elif seq_source.value == 'FASTA File':
            if os.path.exists(fasta_path.value):
                record = next(SeqIO.parse(fasta_path.value, 'fasta'))
                test_protein = str(record.seq)
                seq_id = record.id
                print(f"✓ Loaded from FASTA: {seq_id}")
            else:
                print(f"❌ File not found: {fasta_path.value}")
                return
        
        if test_protein:
            print(f"Length: {len(test_protein)} amino acids")
            print(f"Sequence: {test_protein[:80]}..." if len(test_protein) > 80 else f"Sequence: {test_protein}")

load_seq_button.on_click(load_sequence)

display(seq_source)
display(HTML('<b>Option 1: Example Sequences</b>'))
display(example_dropdown)
display(HTML('<b>Option 2: Direct Input</b>'))
display(direct_input)
display(HTML('<b>Option 3: FASTA File</b>'))
display(fasta_path)
display(load_seq_button)
display(seq_status)

## 3. Analyze Original Sequence Concepts

View the biological concepts of the original sequence:

In [ ]:
if model is None or test_protein is None:
    print("❌ Please load model and sequence first")
else:
    # Get concepts
    with torch.inference_mode():
        original_concepts = model.sequences_to_concepts([test_protein])[-1][0]
    
    concept_names = model.list_supported_concept()
    
    # Create DataFrame
    concepts_df = pd.DataFrame({
        'Concept': concept_names,
        'Value': original_concepts.cpu().numpy()
    })
    concepts_df = concepts_df.sort_values('Value', ascending=False)
    
    print(f"Concept Analysis for: {seq_id}")
    print("="*60)
    
    # Show top and bottom concepts
    print("\nTop 10 Concepts (Highest Values):")
    display(concepts_df.head(10))
    
    print("\nBottom 10 Concepts (Lowest Values):")
    display(concepts_df.tail(10))
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Top concepts
    top_10 = concepts_df.head(10)
    axes[0].barh(range(len(top_10)), top_10['Value'].values, color='green', alpha=0.7)
    axes[0].set_yticks(range(len(top_10)))
    axes[0].set_yticklabels(top_10['Concept'].values)
    axes[0].set_xlabel('Concept Value')
    axes[0].set_title('Top 10 Concepts')
    axes[0].grid(True, alpha=0.3, axis='x')
    
    # Bottom concepts
    bottom_10 = concepts_df.tail(10)
    axes[1].barh(range(len(bottom_10)), bottom_10['Value'].values, color='red', alpha=0.7)
    axes[1].set_yticks(range(len(bottom_10)))
    axes[1].set_yticklabels(bottom_10['Concept'].values)
    axes[1].set_xlabel('Concept Value')
    axes[1].set_title('Bottom 10 Concepts')
    axes[1].grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()
    
    # Store for later use
    original_concepts_df = concepts_df

## 4. Configure and Run Intervention

Select the concept to modify and intervention parameters:

In [ ]:
if model is None:
    print("❌ Please load model first")
else:
    # Available concepts
    all_concepts = model.list_supported_concept()
    
    # Concept selection
    concept_widget = widgets.Dropdown(
        options=all_concepts,
        value='gravy',
        description='Concept:',
        layout=widgets.Layout(width='50%')
    )
    
    # Intervention type
    intervention_type = widgets.RadioButtons(
        options=['positive', 'negative'],
        value='negative',
        description='Direction:'
    )
    
    # Number of edits
    num_edits = widgets.IntSlider(
        value=5,
        min=1,
        max=20,
        step=1,
        description='Num Edits:',
        continuous_update=False
    )
    
    # Number of iterations
    num_iterations = widgets.IntSlider(
        value=1,
        min=1,
        max=10,
        step=1,
        description='Iterations:',
        continuous_update=False
    )
    
    # Run button
    run_button = widgets.Button(
        description='Run Intervention',
        button_style='success',
        icon='play'
    )
    
    intervention_output = widgets.Output()
    
    # Store results
    intervention_results = []
    
    def run_intervention(b):
        global intervention_results
        with intervention_output:
            clear_output()
            
            if test_protein is None:
                print("❌ Please load a sequence first")
                return
            
            print(f"Running intervention...")
            print(f"  Concept: {concept_widget.value}")
            print(f"  Direction: {intervention_type.value}")
            print(f"  Edits per iteration: {num_edits.value}")
            print(f"  Iterations: {num_iterations.value}")
            print()
            
            try:
                intervention_results = []
                current_seq = test_protein
                
                for iteration in range(num_iterations.value):
                    print(f"Iteration {iteration + 1}/{num_iterations.value}...")
                    
                    [new_seq] = model.intervene_on_sequences(
                        [current_seq],
                        concept_widget.value,
                        edits=num_edits.value,
                        intervention_type=intervention_type.value
                    )
                    
                    # Compute metrics
                    edit_distance = Levenshtein.distance(test_protein, new_seq)
                    
                    # Get new concepts
                    with torch.inference_mode():
                        new_concepts = model.sequences_to_concepts([new_seq])[-1][0]
                    
                    concept_idx = all_concepts.index(concept_widget.value)
                    concept_change = new_concepts[concept_idx].item() - original_concepts[concept_idx].item()
                    
                    intervention_results.append({
                        'iteration': iteration + 1,
                        'sequence': new_seq,
                        'edit_distance': edit_distance,
                        'concept_value': new_concepts[concept_idx].item(),
                        'concept_change': concept_change,
                        'all_concepts': new_concepts
                    })
                    
                    print(f"  Edit distance from original: {edit_distance}")
                    print(f"  Concept change: {concept_change:+.4f}")
                    print()
                    
                    current_seq = new_seq
                
                print("✓ Intervention complete!")
                print(f"\nFinal Results:")
                print(f"  Total edits: {intervention_results[-1]['edit_distance']}")
                print(f"  Total concept change: {intervention_results[-1]['concept_change']:+.4f}")
                
            except Exception as e:
                print(f"❌ Error during intervention: {e}")
                import traceback
                traceback.print_exc()
    
    run_button.on_click(run_intervention)
    
    print("Intervention Configuration:")
    print("\nConcept descriptions:")
    print("  • gravy: Grand Average of Hydropathy (hydrophobicity)")
    print("  • aromaticity: Proportion of aromatic amino acids")
    print("  • instability_index: Protein stability measure")
    print("  • isoelectric_point: pH at which protein has no net charge")
    print("  • molecular_weight: Molecular mass in Daltons")
    print("\nIntervention directions:")
    print("  • positive: Increase the concept value")
    print("  • negative: Decrease the concept value")
    print()
    
    display(concept_widget)
    display(intervention_type)
    display(num_edits)
    display(num_iterations)
    display(run_button)
    display(intervention_output)

## 5. Compare Original and Modified Sequences

Analyze the changes made by the intervention:

In [ ]:
if not intervention_results:
    print("❌ Please run an intervention first")
else:
    final_result = intervention_results[-1]
    modified_seq = final_result['sequence']
    
    print("Sequence Comparison")
    print("="*80)
    print(f"Original ({len(test_protein)} aa):")
    print(f"  {test_protein}")
    print()
    print(f"Modified ({len(modified_seq)} aa):")
    print(f"  {modified_seq}")
    print()
    print(f"Edit Distance: {final_result['edit_distance']}")
    print(f"Sequence Identity: {100 * (1 - final_result['edit_distance']/len(test_protein)):.2f}%")
    print()
    
    # Show differences
    print("Sequence Alignment (showing first 200 positions):")
    max_display = min(200, len(test_protein))
    diff_line = ''
    for i in range(max_display):
        if i < len(modified_seq) and test_protein[i] == modified_seq[i]:
            diff_line += '|'
        else:
            diff_line += 'X'
    
    print(f"Original: {test_protein[:max_display]}")
    print(f"          {diff_line}")
    print(f"Modified: {modified_seq[:max_display]}")
    print()
    
    # List specific changes
    changes = []
    for i, (orig, mod) in enumerate(zip(test_protein, modified_seq)):
        if orig != mod:
            changes.append((i+1, orig, mod))
    
    print(f"Specific mutations ({len(changes)} total):")
    for pos, orig, mod in changes[:20]:  # Show first 20
        print(f"  Position {pos}: {orig} → {mod}")
    if len(changes) > 20:
        print(f"  ... and {len(changes) - 20} more")

## 6. Compute and Compare Biological Metrics

Calculate biological properties of both sequences:

In [ ]:
if not intervention_results:
    print("❌ Please run an intervention first")
else:
    def compute_bio_metrics(seq: str) -> dict[str, float]:
        """Compute biological metrics for a sequence."""
        try:
            analyzer = ProteinAnalysis(seq)
            ss = analyzer.secondary_structure_fraction()
            return {
                'molecular_weight': analyzer.molecular_weight(),
                'aromaticity': analyzer.aromaticity(),
                'instability_index': analyzer.instability_index(),
                'isoelectric_point': analyzer.isoelectric_point(),
                'gravy': analyzer.gravy(),
                'helix_fraction': ss[0],
                'turn_fraction': ss[1],
                'sheet_fraction': ss[2],
            }
        except Exception as e:
            print(f"Warning: Error computing metrics: {e}")
            return {}
    
    # Compute metrics
    original_metrics = compute_bio_metrics(test_protein)
    modified_metrics = compute_bio_metrics(intervention_results[-1]['sequence'])
    
    # Create comparison DataFrame
    comparison_df = pd.DataFrame({
        'Property': list(original_metrics.keys()),
        'Original': list(original_metrics.values()),
        'Modified': list(modified_metrics.values())
    })
    comparison_df['Change'] = comparison_df['Modified'] - comparison_df['Original']
    comparison_df['% Change'] = 100 * comparison_df['Change'] / (comparison_df['Original'] + 1e-10)
    
    print("Biological Properties Comparison")
    print("="*80)
    display(comparison_df)
    
    # Visualize changes
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Bar plot of key properties
    key_props = ['molecular_weight', 'aromaticity', 'instability_index', 'gravy']
    key_data = comparison_df[comparison_df['Property'].isin(key_props)]
    
    x = np.arange(len(key_data))
    width = 0.35
    
    axes[0, 0].bar(x - width/2, key_data['Original'], width, label='Original', alpha=0.8)
    axes[0, 0].bar(x + width/2, key_data['Modified'], width, label='Modified', alpha=0.8)
    axes[0, 0].set_xlabel('Property')
    axes[0, 0].set_ylabel('Value')
    axes[0, 0].set_title('Key Properties Comparison')
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(key_data['Property'], rotation=45, ha='right')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3, axis='y')
    
    # Secondary structure comparison
    ss_props = ['helix_fraction', 'turn_fraction', 'sheet_fraction']
    ss_data = comparison_df[comparison_df['Property'].isin(ss_props)]
    
    x = np.arange(len(ss_data))
    axes[0, 1].bar(x - width/2, ss_data['Original'], width, label='Original', alpha=0.8)
    axes[0, 1].bar(x + width/2, ss_data['Modified'], width, label='Modified', alpha=0.8)
    axes[0, 1].set_xlabel('Structure Type')
    axes[0, 1].set_ylabel('Fraction')
    axes[0, 1].set_title('Secondary Structure Comparison')
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels(['Helix', 'Turn', 'Sheet'])
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3, axis='y')
    
    # Percentage change
    axes[1, 0].barh(range(len(comparison_df)), comparison_df['% Change'],
                    color=['red' if x < 0 else 'green' for x in comparison_df['% Change']],
                    alpha=0.7)
    axes[1, 0].set_yticks(range(len(comparison_df)))
    axes[1, 0].set_yticklabels(comparison_df['Property'])
    axes[1, 0].set_xlabel('% Change')
    axes[1, 0].set_title('Percentage Change in Properties')
    axes[1, 0].axvline(x=0, color='black', linestyle='-', linewidth=0.8)
    axes[1, 0].grid(True, alpha=0.3, axis='x')
    
    # Concept value trajectory (if multiple iterations)
    if len(intervention_results) > 1:
        iterations = [r['iteration'] for r in intervention_results]
        concept_values = [r['concept_value'] for r in intervention_results]
        edit_distances = [r['edit_distance'] for r in intervention_results]
        
        ax1 = axes[1, 1]
        ax2 = ax1.twinx()
        
        color1 = 'tab:blue'
        ax1.plot(iterations, concept_values, 'o-', color=color1, linewidth=2, markersize=8)
        ax1.set_xlabel('Iteration')
        ax1.set_ylabel('Concept Value', color=color1)
        ax1.tick_params(axis='y', labelcolor=color1)
        ax1.grid(True, alpha=0.3)
        
        color2 = 'tab:orange'
        ax2.plot(iterations, edit_distances, 's-', color=color2, linewidth=2, markersize=8)
        ax2.set_ylabel('Edit Distance', color=color2)
        ax2.tick_params(axis='y', labelcolor=color2)
        
        axes[1, 1].set_title('Intervention Trajectory')
    else:
        axes[1, 1].text(0.5, 0.5, 'Run multiple iterations\nto see trajectory',
                       ha='center', va='center', fontsize=12)
        axes[1, 1].axis('off')
    
    plt.tight_layout()
    plt.show()

## 7. Compare All Concepts

See how all concepts changed:

In [ ]:
if not intervention_results:
    print("❌ Please run an intervention first")
else:
    concept_names = model.list_supported_concept()
    modified_concepts = intervention_results[-1]['all_concepts'].cpu().numpy()
    
    # Create comparison DataFrame
    concepts_comparison = pd.DataFrame({
        'Concept': concept_names,
        'Original': original_concepts.cpu().numpy(),
        'Modified': modified_concepts
    })
    concepts_comparison['Change'] = concepts_comparison['Modified'] - concepts_comparison['Original']
    concepts_comparison['Abs_Change'] = concepts_comparison['Change'].abs()
    
    # Sort by absolute change
    concepts_comparison = concepts_comparison.sort_values('Abs_Change', ascending=False)
    
    print("Top 20 Concepts by Absolute Change")
    print("="*80)
    display(concepts_comparison.head(20))
    
    # Visualize top changes
    top_changes = concepts_comparison.head(15)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    y_pos = np.arange(len(top_changes))
    colors = ['red' if x < 0 else 'green' for x in top_changes['Change']]
    
    ax.barh(y_pos, top_changes['Change'], color=colors, alpha=0.7)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top_changes['Concept'])
    ax.set_xlabel('Concept Change')
    ax.set_title('Top 15 Concept Changes After Intervention')
    ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()

## 8. Export Results

Save intervention results to files:

In [ ]:
# Export options
output_dir = widgets.Text(
    value='outputs/intervention_results',
    description='Output Dir:',
    layout=widgets.Layout(width='50%')
)

export_button = widgets.Button(
    description='Export Results',
    button_style='success',
    icon='save'
)

export_status = widgets.Output()

def export_intervention_results(b):
    with export_status:
        clear_output()
        
        if not intervention_results:
            print("❌ No results to export")
            return
        
        # Create output directory
        os.makedirs(output_dir.value, exist_ok=True)
        print(f"Exporting to: {output_dir.value}")
        
        # Save sequences
        with open(f"{output_dir.value}/sequences.txt", 'w') as f:
            f.write(f"Original Sequence:\n{test_protein}\n\n")
            f.write(f"Modified Sequence:\n{intervention_results[-1]['sequence']}\n")
        print("✓ Saved sequences.txt")
        
        # Save metrics comparison
        if 'comparison_df' in globals():
            comparison_df.to_csv(f"{output_dir.value}/metrics_comparison.csv", index=False)
            print("✓ Saved metrics_comparison.csv")
        
        # Save concepts comparison
        if 'concepts_comparison' in globals():
            concepts_comparison.to_csv(f"{output_dir.value}/concepts_comparison.csv", index=False)
            print("✓ Saved concepts_comparison.csv")
        
        # Save intervention trajectory
        trajectory_df = pd.DataFrame(intervention_results)
        trajectory_df = trajectory_df.drop('all_concepts', axis=1)  # Drop tensor column
        trajectory_df.to_csv(f"{output_dir.value}/intervention_trajectory.csv", index=False)
        print("✓ Saved intervention_trajectory.csv")
        
        # Save summary
        with open(f"{output_dir.value}/summary.txt", 'w') as f:
            f.write(f"Intervention Summary\n")
            f.write(f"="*60 + "\n\n")
            f.write(f"Original Sequence ID: {seq_id}\n")
            f.write(f"Target Concept: {concept_widget.value}\n")
            f.write(f"Intervention Type: {intervention_type.value}\n")
            f.write(f"Edits per Iteration: {num_edits.value}\n")
            f.write(f"Total Iterations: {num_iterations.value}\n\n")
            f.write(f"Results:\n")
            f.write(f"  Total Edit Distance: {intervention_results[-1]['edit_distance']}\n")
            f.write(f"  Concept Change: {intervention_results[-1]['concept_change']:+.4f}\n")
            f.write(f"  Sequence Identity: {100 * (1 - intervention_results[-1]['edit_distance']/len(test_protein)):.2f}%\n")
        print("✓ Saved summary.txt")
        
        print("\n✓ Export complete!")

export_button.on_click(export_intervention_results)

display(output_dir)
display(export_button)
display(export_status)